# 🌌 TabPFN-3 Stacker — diverse bases → logits + raw features → TabPFN-3
Reproduces the current S6E6 meta (Chris Deotte's stacker + philippsinger's TabPFN-3 version):
each base model's OOF/test probabilities → **logits**, append the **raw features**, and let
**TabPFN-3** be the meta-model. Adds two *distinct* RealMLP bases of our own (small-batch +
high-label-smoothing) that an offline screen showed are less redundant with the pool.

**Setup:** GPU (T4×2 ideal), Internet **On**; inputs = competition data + the
`s6e6-stack-bases` dataset.

## 1. Install TabPFN

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tabpfn"], check=True)
print("installed tabpfn")

## 2. Imports + device + TabPFN-3 weights

In [ ]:
import os, glob, numpy as np, pandas as pd, torch
from sklearn.metrics import balanced_accuracy_score
# Use the TabPFN-3 weights from the added model input (avoids the gated HuggingFace download +
# license prompt, which has no interactive terminal in a batch kernel). Point the cache dir at
# the mounted model so .fit() finds the weights locally.
_w = (glob.glob("/kaggle/input/**/tabpfn-3/**/*.ckpt", recursive=True)
      or glob.glob("/kaggle/input/**/tabpfn*3*/**/*.ckpt", recursive=True)
      or glob.glob("/kaggle/input/**/*.ckpt", recursive=True))
if _w:
    os.environ["TABPFN_MODEL_CACHE_DIR"] = os.path.dirname(_w[0])
    print("TABPFN_MODEL_CACHE_DIR =", os.environ["TABPFN_MODEL_CACHE_DIR"])
else:
    print("WARNING: TabPFN-3 .ckpt not found under /kaggle/input — add the prior-labsai/tabpfn-3 model input")
ndev = torch.cuda.device_count()
DEVICE = [f"cuda:{i}" for i in range(ndev)] if ndev > 1 else ("cuda" if ndev == 1 else "cpu")
print("CUDA devices:", ndev, "->", DEVICE)
ROOT = "/kaggle/input"; OUT = "/kaggle/working"
EPS, CLIP = 1e-15, 30.0
TMAP = {"GALAXY": 0, "QSO": 1, "STAR": 2}; CLASSES = ["GALAXY", "QSO", "STAR"]
def logit(p):
    p = np.clip(p, EPS, 1 - EPS).astype(np.float64)
    return np.clip(np.log(p / (1 - p)), -CLIP, CLIP)

## 3. Load data + pre-normalized base predictions

In [ ]:
comp = glob.glob(f"{ROOT}/**/train.csv", recursive=True)[0].rsplit("/", 1)[0]
train = pd.read_csv(f"{comp}/train.csv"); test = pd.read_csv(f"{comp}/test.csv")
y = train["class"].map(TMAP).to_numpy(); test_ids = test["id"]; N, M_ = len(train), len(test)
BASEDIR = glob.glob(f"{ROOT}/**/oof_xgb0.npy", recursive=True)[0].rsplit("/", 1)[0]
print("train", N, "test", M_, "| bases:", BASEDIR)

# 6 public diverse bases + our 2 distinct RealMLP bases (all clean (N,3)/(M,3) arrays)
BASES = ["xgb0", "xgb1", "realmlp0", "realmlp1", "tabm", "cat", "bs128", "ls008"]
oof_parts, test_parts = [], []
for nm in BASES:
    o = np.load(f"{BASEDIR}/oof_{nm}.npy").astype(np.float64)
    t = np.load(f"{BASEDIR}/test_{nm}.npy").astype(np.float64)
    ba = balanced_accuracy_score(y, o.argmax(1))
    assert o.shape == (N, 3) and t.shape == (M_, 3), f"{nm} shape {o.shape}/{t.shape}"
    assert ba > 0.90, f"{nm}: solo BA {ba:.3f} too low"
    print(f"  {nm:9s} BA={ba:.4f}")
    oof_parts.append(logit(o)); test_parts.append(logit(t))

feat = ["alpha", "delta", "u", "g", "r", "i", "z", "redshift"]
Xtr = np.hstack(oof_parts + [train[feat].to_numpy(np.float64)]).astype(np.float32)
Xte = np.hstack(test_parts + [test[feat].to_numpy(np.float64)]).astype(np.float32)
print(f"stack matrix: {len(BASES)} bases x3 logits + {len(feat)} raw = {Xtr.shape[1]} cols")

## 4. TabPFN-3 meta (fit on all train, predict test)

In [ ]:
from tabpfn import TabPFNClassifier

def fit_predict(subsample=None):
    idx = np.arange(N) if not subsample else np.random.RandomState(42).choice(N, subsample, replace=False)
    clf = TabPFNClassifier(device=DEVICE, n_estimators=2, balance_probabilities=True, ignore_pretraining_limits=True)
    clf.fit(Xtr[idx], y[idx])
    return clf.predict_proba(Xte)

try:
    test_prob = fit_predict()
except RuntimeError as e:                      # OOM on a single small GPU -> cap context
    print("retrying with subsampled context:", e)
    torch.cuda.empty_cache(); test_prob = fit_predict(subsample=200_000)
print("test_prob", test_prob.shape)

## 5. Submit

In [ ]:
preds = np.asarray(CLASSES)[test_prob.argmax(1)]
pd.DataFrame({"id": test_ids, "class": preds}).to_csv(f"{OUT}/submission.csv", index=False)
np.save(f"{OUT}/test_stack_tabpfn.npy", test_prob)
print("wrote submission.csv + test_stack_tabpfn.npy")
print("pred class dist:", dict(zip(*np.unique(preds, return_counts=True))))